In [4]:
def transform_staff_count(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    text_cols = [
        "План/Факт",
        "Подразделение",
        "Отдел",
        "ФИО работника",
        "Должность/профессияработника",
    ]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].astype("string")

    df["Дата начала работы"] = pd.to_datetime(
        df["Дата начала работы"], errors="coerce"
    ).dt.date
    df["Дата окончания работы"] = pd.to_datetime(
        df["Дата окончания работы"], errors="coerce"
    ).dt.date

    df["Дата"] = df.apply(
        lambda row: _expand_dates(
            row["Дата начала работы"],
            row["Дата окончания работы"],
        ),
        axis=1,
    )

    df = df.explode("Дата", ignore_index=True)
    df = df[df["Дата"].notna()].copy()
    df["Дата"] = pd.to_datetime(df["Дата"], errors="coerce").dt.date

    df.drop(columns=["Год", "Пол", "Основная профессия при приеме"], errors="ignore", inplace=True)

    df = df.rename(
        columns={
            "Отдел": "ЦФО",
            "ФИО работника": "ФИО",
            "Должность/профессияработника": "Профессия",
        }
    )

    return df

In [7]:
df_plan = transform_staff_count(
    pd.read_excel("./source/План_численность.xlsx", sheet_name="Лист1")
)

df_fact = transform_staff_count(
    pd.read_excel("./source/Факт_численность.xlsx", sheet_name="Лист1")
)